# Entrenamiento de Modelos (Baseline)

En este notebook vamos a cargar el dataset limpio de vehículos, preprocesar los datos y entrenar tres modelos diferentes (Linear Regression, Random Forest y XGBoost). Todo el proceso será registrado y versionado utilizando **MLflow**.

In [11]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from mlflow.tracking import MlflowClient

# Configurar MLflow local
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("ValorAuto_Model_Comparison")
print("MLflow preparado (tracking uri: local sqlite).")

MLflow preparado (tracking uri: local sqlite).


## Carga de Datos y Preprocesamiento

Cargaremos el dataset `vehicles_clean.csv`, definiremos las variables predictoras (features) y configuraremos un pipeline de `scikit-learn` para manejar valores nulos y codificación *One-Hot*.

In [12]:
data_path = '../data/vehicles/vehicles_clean.csv'

if not os.path.exists(data_path):
    print(f"¡Atención! No se encontró el dataset en {data_path}.")
else:
    print(f"Cargando dataset desde {data_path}...")
    df = pd.read_csv(data_path, on_bad_lines='skip', engine='python')

    # Selección de características (features) relevantes y variable objetivo (target)
    features = ['year', 'odometer', 'manufacturer', 'fuel', 'transmission', 'drive', 'type']
    target = 'price'

    # Filtrar solo las columnas seleccionadas y eliminar filas sin target
    df_ml = df[features + [target]].dropna(subset=[target])

    X = df_ml[features]
    y = df_ml[target]

    # Identificar columnas por tipo
    num_cols = ['year', 'odometer']
    cat_cols = ['manufacturer', 'fuel', 'transmission', 'drive', 'type']

    # Preprocesamiento: Imputación + Encoding
    num_transformer = SimpleImputer(strategy='median')
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_cols),
            ('cat', cat_transformer, cat_cols)
        ]
    )

    # SEPARACIÓN EN TRAIN Y TEST
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Dataset cargado exitosamente:")
    print(f"Muestras de entrenamiento (Train): {X_train.shape[0]}")
    print(f"Muestras de prueba (Test): {X_test.shape[0]}")

Cargando dataset desde ../data/vehicles/vehicles_clean.csv...
Dataset cargado exitosamente:
Muestras de entrenamiento (Train): 80500
Muestras de prueba (Test): 20125


## Definición de Modelos

Vamos a probar 3 enfoques para comparar su rendimiento.

In [13]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

# Diccionario con los modelos a entrenar
models = {
    "Baseline_LinearRegression": {
        "model": LinearRegression(),
        "params": {}
    },
    "Model_RandomForest": {
        "model": RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
        "params": {"n_estimators": 50, "max_depth": 10, "random_state": 42}
    },
    "Model_XGBoost": {
        "model": XGBRegressor(n_estimators=100, max_depth=8, learning_rate=0.1, random_state=42, n_jobs=-1),
        "params": {"n_estimators": 100, "max_depth": 8, "learning_rate": 0.1, "random_state": 42}
    }
}

## Bucle de Entrenamiento y Registro en MLflow

Por cada modelo definido, ejecutaremos el pipeline, predeciremos, evaluaremos y enviaremos todos los datos al servidor local de MLflow.

In [14]:
for model_name, config in models.items():
    with mlflow.start_run(run_name=model_name):
        print(f"Entrenando {model_name}...")

        # Crear pipeline completo
        clf = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', config["model"])])

        # Entrenamiento
        clf.fit(X_train, y_train)

        # Predicciones
        y_pred = clf.predict(X_test)

        # Métricas
        rmse, mae, r2 = eval_metrics(y_test, y_pred)

        # Log a MLflow
        mlflow.log_params(config["params"])
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        mlflow.sklearn.log_model(
            sk_model=clf,
            artifact_path="model",
            serialization_format="cloudpickle"
        )

        print(f"[{model_name}] -> RMSE: {rmse:.2f} | MAE: {mae:.2f} | R2: {r2:.4f}\n")

Entrenando Baseline_LinearRegression...


2026/09/16 23:08:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/16 23:08:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Baseline_LinearRegression] -> RMSE: 8367.85 | MAE: 5737.85 | R2: 0.6484

Entrenando Model_RandomForest...


2026/09/16 23:08:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/16 23:08:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Model_RandomForest] -> RMSE: 7004.02 | MAE: 4677.46 | R2: 0.7536

Entrenando Model_XGBoost...


2026/09/16 23:09:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/16 23:09:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Model_XGBoost] -> RMSE: 6247.01 | MAE: 4005.52 | R2: 0.8040



## Promoción a Producción del Mejor Modelo

Buscamos el modelo con el menor RMSE y lo registramos formalmente en el *Model Registry* con la etiqueta **Production**.

In [15]:
client = MlflowClient()
experiment = mlflow.get_experiment_by_name("ValorAuto_Model_Comparison")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse ASC"]
)

if not runs.empty:
    best_run = runs.iloc[0]
    run_id = best_run.run_id
    model_uri = f"runs:/{run_id}/model"
    model_name_registry = "ValorAuto_Model"
    
    print(f"Registrando el mejor modelo (Run ID: {run_id}) en el Model Registry...")
    try:
        registered_model = mlflow.register_model(model_uri=model_uri, name=model_name_registry)
        client.transition_model_version_stage(
            name=model_name_registry,
            version=registered_model.version,
            stage="Production"
        )
        print(f"Modelo versión {registered_model.version} asignado a Production exitosamente.")
    except Exception as e:
        print(f"Nota: No se pudo registrar en Production. Detalles: {e}")

Registered model 'ValorAuto_Model' already exists. Creating a new version of this model...
2026/09/16 23:09:09 WARNING mlflow.tracking._model_registry.fluent: Run with id 4bae187072d24131afecb23cf9ebe7ca has no artifacts at artifact path 'model', registering model based on models:/m-0605233de31b45f0a69cfa4c6334d36e instead


Registrando el mejor modelo (Run ID: 4bae187072d24131afecb23cf9ebe7ca) en el Model Registry...
Modelo versión 3 asignado a Production exitosamente.


Created version '3' of model 'ValorAuto_Model'.
C:\Users\LEONGO.037\AppData\Local\Temp\ipykernel_22052\439518426.py:17: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
